# ClipCap subset training on Google Colab

Mục tiêu là train độc lập các subset Flickr8k `1%`, `5%`, `10%`, `25%`, `100%` trong cùng số epoch cố định để so sánh ảnh hưởng của lượng dữ liệu. Mỗi subset khởi tạo mapper mới, dùng GPT-2 freeze, FP32, learning rate cố định và checkpoint riêng trên Google Drive.

## 1. Kết nối GPU và lấy mã nguồn

Trong Colab, chọn `Runtime > Change runtime type > GPU` trước khi chạy. Nếu nhánh của nhóm thay đổi, sửa biến `BRANCH`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

import torch

assert torch.cuda.is_available(), "Hãy bật GPU runtime trước khi training"
print("GPU:", torch.cuda.get_device_name(0))

REPO_URL = "https://github.com/HnhanBk415/zfs-clip-image-captioning.git"
BRANCH = "refactor/huuthien/Model"
PROJECT_DIR = Path("/content/zfs-clip-image-captioning")

if not PROJECT_DIR.is_dir():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO_URL, str(PROJECT_DIR)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)

os.chdir(PROJECT_DIR)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True,
)
print("Project directory:", PROJECT_DIR)

## 2. Gắn Google Drive

Checkpoint được ghi trực tiếp vào Drive. CLIP features và tokenized tensors được cache trên Drive nhưng được copy về `/content` trước khi train để tránh đọc nhiều lần qua Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/clipcap_colab")
DATA_CACHE_DIR = DRIVE_ROOT / "data_cache"
CHECKPOINT_ROOT = DRIVE_ROOT / "experiments_fixed_epoch"
DATA_CACHE_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
print("Checkpoint root:", CHECKPOINT_ROOT)

## 3. Khôi phục hoặc tạo dữ liệu trung gian

Lần đầu tiên, cell này tải Flickr8k, trích xuất CLIP features và tokenize caption. Các lần sau, nó khôi phục các file `.pt` từ Drive.

In [ ]:
import shutil

from src.clipcap.preprocessing.clip_features import run_clip_feature_extraction
from src.clipcap.preprocessing.tokenization import main as run_tokenization
from src.config.clipcap_config import (
    CLIPCAP_TRAIN_SUBSETS,
    FEATURE_DIR,
    TOKENIZED_DIR,
    create_clipcap_fixed_epoch_config,
)

feature_path = FEATURE_DIR / "clip_features.pt"
token_filenames = [
    *(f"{subset_name}.pt" for subset_name in CLIPCAP_TRAIN_SUBSETS),
    "val.pt",
    "test.pt",
]

cached_feature_path = DATA_CACHE_DIR / "features" / feature_path.name
if not feature_path.is_file() and cached_feature_path.is_file():
    feature_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(cached_feature_path, feature_path)

for filename in token_filenames:
    local_path = TOKENIZED_DIR / filename
    cached_path = DATA_CACHE_DIR / "tokenized" / filename
    if not local_path.is_file() and cached_path.is_file():
        local_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(cached_path, local_path)

if not feature_path.is_file():
    run_clip_feature_extraction()

if not all((TOKENIZED_DIR / filename).is_file() for filename in token_filenames):
    run_tokenization()

cached_feature_path.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(feature_path, cached_feature_path)
for filename in token_filenames:
    cached_path = DATA_CACHE_DIR / "tokenized" / filename
    cached_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(TOKENIZED_DIR / filename, cached_path)

print("CLIP feature:", feature_path)
print("Tokenized files:", len(token_filenames))

## 4. Cấu hình training

Toàn bộ hyperparameter được lấy từ `src/config/clipcap_config.py`. Notebook chỉ đổi nơi lưu checkpoint sang Google Drive. `final.pt` tại epoch cố định là checkpoint chính thức; `best.pt` được giữ để tham khảo và `latest.pt` chứa thêm optimizer để resume.

In [ ]:
BASE_CONFIG = create_clipcap_fixed_epoch_config(
    output_root=CHECKPOINT_ROOT,
)
BASE_CONFIG

## 5. Train tuần tự các subset

Mỗi subset khởi tạo model mới và luôn chạy đủ số epoch cố định; validation loss vẫn được theo dõi nhưng không kích hoạt early stopping. Nếu Colab bị ngắt, chạy lại notebook và cell này; thí nghiệm hoàn thành sẽ được bỏ qua, thí nghiệm dở dang sẽ tiếp tục từ `latest.pt`.

In [ ]:
from src.clipcap.training import run_subset_experiments

SUBSETS_TO_TRAIN = CLIPCAP_TRAIN_SUBSETS

results = run_subset_experiments(
    BASE_CONFIG,
    subset_names=SUBSETS_TO_TRAIN,
    device="cuda",
    resume=True,
    skip_completed=True,
    show_progress=True,
)

## 6. Kiểm tra kết quả training

In [ ]:
import pandas as pd

summary_path = CHECKPOINT_ROOT / "experiment_summary.csv"
summary = pd.read_csv(summary_path)
display(summary)

for result in results:
    final_path = Path(result["official_checkpoint"])
    assert final_path.is_file(), f"Missing checkpoint: {final_path}"
    assert result["final_epoch"] == BASE_CONFIG.max_epochs
print("All fixed-epoch subset checkpoints are available.")